In [ ]:
###test frames_and_faces_preprocessing


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#@title Install Requirements

# test frames and face extraction with mediapipe face detector
!pip install mediapipe==0.10.20
# install mediapy to show video in colab demo
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy
print("done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 12.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled p

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 6.0 MB/s eta 0:00:00
done!


In [1]:
# you might need to restart the session to use mediapipe on colab
# run this cell to make sure the correct version has been installed
import mediapipe as mp
print(mp.__version__) # should be 0.10.20

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(


0.10.20


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
import os
import sys
import timm # for using the XceptionNet model (pretrained) # pip install timm
import yaml
import glob
import matplotlib.pyplot as plt
# from scipy.special import expit
import glob
import numpy as np
import cv2
# import mediapy as media
import mediapipe as mp
from PIL import Image
# from demo_utils import *

In [7]:
from genericpath import exists
import os
import mediapipe as mp
import cv2
# import numpy as np
import re
import json

# mediapipe face detection model
face_detection_model = mp.solutions.face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5)
# model_selection=0 -> use the short model for the face detection
# min_detection_confidence=0.5 -> min confidence threshold to detect a face

# make the facedetector run in CPU mode

# ----------------------------------------- #
# # mp face detector
# BaseOptions = mp.tasks.BaseOptions
# FaceDetector = mp.tasks.vision.FaceDetector
# FaceDetectorOptions = mp.tasks.vision.FaceDetectorOptions
# VisionRunningMode = mp.tasks.vision.RunningMode

# # Create an FaceDetector object.
# face_base_options = BaseOptions(model_asset_path='./face_landmark_model/blaze_face_short_range.tflite')
# face_options = FaceDetectorOptions(
#     base_options=face_base_options,
#     running_mode=VisionRunningMode.VIDEO,
#     min_detection_confidence=0.88, # min confidence threshold to detect a face (fine-tuned)
#     min_suppression_threshold=0.3 # min non-max suppression threshold to detect multiple faces in an image
#     )
# # with FaceDetector.create_from_options(options) as detector:
# face_detection_model = FaceDetector.create_from_options(face_options)
# ----------------------------------------- #

def sort_paths(path):
    # given a path to a folder, sort the files in the folder by the number after 'hand_occlusion' or 'obj_occlusion' in the folder
    path = sorted(path, key=lambda x: int(re.search(r'\d+', x).group()))
    return path


def get_file_name(path):
    return os.path.basename(path)

def create_subfolders(path):
    # create a subfolder for each challenge
    os.makedirs(path + '/hand_occlusion_1', exist_ok=True)
    os.makedirs(path + '/hand_occlusion_2', exist_ok=True)
    os.makedirs(path + '/hand_occlusion_3', exist_ok=True)
    os.makedirs(path + '/obj_occlusion_1', exist_ok=True)
    os.makedirs(path + '/obj_occlusion_2', exist_ok=True)
    os.makedirs(path + '/obj_occlusion_3', exist_ok=True)

def create_subfolders_demo(path):
    # create the subfolder for the demo
    os.makedirs(path + '/hand_occlusion')
    os.makedirs(path + '/obj_occlusion')

# function to extract all the subfolders in a directory
def extract_subfolders(directory):
    # return subfolders following the order of the subfolders in the directory (not sorted)
    # follows the alphabetical order of the subfolders in the directory
    subfolders = [f.path for f in os.scandir(directory) if f.is_dir()]
    return subfolders

# function to extract all the files in a directory (following the order of the files in the directory)
def extract_files(directory):
    all_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            all_files.append(os.path.join(root, file))
    if all_files:
        all_files = sort_paths(all_files)
        return all_files
    else:
        print(f"no files found in the directory {directory} or its subfolders")
        return False

def face_detection(frame, face_detection_model):
    # convert frame to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    # detect faces in the frame
    results = face_detection_model.process(frame_rgb)
    return results

def extract_face_from_frame(frame, face_bbox):
    # extract face from frame
    x, y, w, h = face_bbox
    face = frame[y:y+h, x:x+w]
    return face


def extract_faces_from_videos(vid_paths, out_paths):

    """
        Extract faces from videos using mediapipe face detection model
        input:
            - vid_paths: list of video paths
            - out_paths: list of output paths where the extracted faces will be saved

        output:
            - the extracted faces are saved to the corresponding output paths

        NOTE: the number of frames extracted from the original videos should be 390
        (fixed, corresponding to 13 seconds of guiding video)


        Bounding Box (bbox) enlargement (gist):
        - if a face has been detected
        - compute the face bbox coordinates:  bboxC = detection.location_data.relative_bounding_box
        - compute the image width and height: ih, iw, _ = image.shape
        Then compute the bbox coordinates in pixels:
            - x, y, w, h computed from the relative bounding box coordinates (0.0 to 1.0)
            - x, y, w, h = int(bboxC.xmin * iw), int(bboxC.ymin * ih), int(bboxC.width * iw), int(bboxC.height * ih)
            - bboxC.xmin * iw -> x coordinate of the top-left corner of the bbox (xmin * image_width)
            - bboxC.ymin * ih -> y coordinate of the top-left corner of the bbox
            - bboxC.width * iw -> width of the bbox
            - bboxC.height * ih -> height of the bbox
        -> x, y, w, h = int(bboxC.xmin * iw), int(bboxC.ymin * ih), int(bboxC.width * iw), int(bboxC.height * ih)

        - enlarge the face bounding box by 30% (15% to the left and 15% to the top of the bbox)
            x = max(0, x - int(0.15 * w))
            y = max(0, y - int(0.15 * h))
            # the 0.15 * w and 0.15 * h are used to go 15% to the left and 15% to the top of the bbox
            w = min(iw, w + int(0.3 * w))
            h = min(ih, h + int(0.3 * h)) # min(ih, h + 0.3 * h) -> height of the bbox

        - extract face from frame: face = extract_face_from_frame(image, (x, y, w, h))
        - the face is then saved to the corresponding output path
        - the process is repeated for all frames in the video (until 390 frames are extracted)
    """

    for vid, out in zip(vid_paths, out_paths):
        print('extracting faces from video:', vid)  # print video name
        print('saving faces to:', out)  # print output path
        # Path to video file
        vidObj = cv2.VideoCapture(vid)
        # Used as counter variable
        count = 0
        # checks whether frames were extracted
        success = 1

        while success and count < 390: # until extracted 390 frames (13 seconds of guiding video)
            # vidObj object calls read function extract frames
            success, image = vidObj.read()
            results = face_detection(image, face_detection_model)
            if results.detections: # if there are faces detected in the frame
                for detection in results.detections:
                    # compute the bbox coordinates
                    bboxC = detection.location_data.relative_bounding_box
                    # get the image width and height
                    ih, iw, _ = image.shape
                    # get the bbox coordinates in pixels
                    x, y, w, h = int(bboxC.xmin * iw), int(bboxC.ymin * ih), int(bboxC.width * iw), int(bboxC.height * ih)

                    # enlarge face bbox by 30%
                    x = max(0, x - int(0.15 * w))
                    y = max(0, y - int(0.15 * h))
                    # the 0.15 * w and 0.15 * h are used to go 15% to the left and 15% to the top of the bbox
                    w = min(iw, w + int(0.3 * w))
                    h = min(ih, h + int(0.3 * h)) # min(ih, h + 0.3 * h) -> height of the bbox

                    # # enlarge face bbox by 30% while keeping the aspect ratio
                    # scale_factor = 1.3
                    # new_w = int(w * scale_factor)
                    # new_h = int(h * scale_factor)
                    # x = max(0, x - (new_w - w) // 2)
                    # y = max(0, y - (new_h - h) // 2)
                    # w = min(iw - x, new_w)
                    # h = min(ih - y, new_h)

                    # extract face from frame
                    face = extract_face_from_frame(image, (x, y, w, h))

                    # check if the frame is in the index of the frames to extract

                    # save face
                    cv2.imwrite(out + "/frame%d.jpg" % count, face)
                    count += 1
        print("done")
        vidObj.release()
        cv2.destroyAllWindows()

def extract_faces_from_videos_json(vid_paths, out_paths, frames_json_path):
    # print("ciao")

    if frames_json_path:
        # load the json file with the index of the frames to extract
        with open(frames_json_path, 'r') as f:
            frames_index = json.load(f)

    else:
        print("No JSON file with the index of the frames to extract")
        exit()

    for vid, out in zip(vid_paths, out_paths):
        print('extracting faces from video:', vid)  # print video name
        print('saving faces to:', out)  # print output path
        # user_id/algo_name/challenge_id/frame%d.jpg -> get the user id, algorithm name and challenge id from the output path
        ## get user info from out_path
        # get info on the output path - use these info to extract frames from json
        user_id = vid.split('/')[-3] # get the user id from the video path
        # algo = out.split('/')[-2] # get the algorithm name from the output path
        challenge_id = vid.split('/')[-2] # get the challenge id from the output path
        print(f"user_id: {user_id} - algo: DLC - challenge_id: {challenge_id}")
        # --------------------------------------- #
        # Path to video file
        vidObj = cv2.VideoCapture(vid)
        # Used as counter variable
        count = 0
        # checks whether frames were extracted
        success = 1


        frames_list = frames_index[user_id][challenge_id]
        # convert the list of frames names (e.g., frame170.jpg) to a list of frame numbers (e.g., 170)
        frames_to_extract = [int(frame.split('frame')[1].split('.jpg')[0]) for frame in frames_list]
        print(f"frames to extract from the json file for user {user_id}, algorithm DLC, challenge {challenge_id}:", frames_to_extract)
        print(frames_to_extract)

        # sanity check - 100 frames per challenge
        if frames_to_extract and len(frames_to_extract) != 100:
            print("ERROR: number of frames to extract from the json file is not correct (!= 100)")
        elif frames_to_extract:
            print("number of frames to extract from the json file is correct (= 100)")


        while success and count < 390: # until extracted 390 frames (13 seconds of guiding video)
            # vidObj object calls read function extract frames
            success, image = vidObj.read()
            results = face_detection(image, face_detection_model)
            if results.detections and count in frames_to_extract: # if there are faces detected in the frame
                for detection in results.detections:
                    # compute the bbox coordinates
                    bboxC = detection.location_data.relative_bounding_box
                    # get the image width and height
                    ih, iw, _ = image.shape
                    # get the bbox coordinates in pixels
                    x, y, w, h = int(bboxC.xmin * iw), int(bboxC.ymin * ih), int(bboxC.width * iw), int(bboxC.height * ih)

                    # enlarge face bbox by 30%
                    x = max(0, x - int(0.15 * w))
                    y = max(0, y - int(0.15 * h))
                    # the 0.15 * w and 0.15 * h are used to go 15% to the left and 15% to the top of the bbox
                    w = min(iw, w + int(0.3 * w))
                    h = min(ih, h + int(0.3 * h)) # min(ih, h + 0.3 * h) -> height of the bbox

                    # # enlarge face bbox by 30% while keeping the aspect ratio
                    # scale_factor = 1.3
                    # new_w = int(w * scale_factor)
                    # new_h = int(h * scale_factor)
                    # x = max(0, x - (new_w - w) // 2)
                    # y = max(0, y - (new_h - h) // 2)
                    # w = min(iw - x, new_w)
                    # h = min(ih - y, new_h)

                    # extract face from frame
                    face = extract_face_from_frame(image, (x, y, w, h))

                    # check if the frame is in the index of the frames to extract

                    if frames_to_extract and count in frames_to_extract: # if the frame is not in the index of the frames to extract, skip it
                        # extract face only for the frames in the frams_to_extract_list
                        print(f"frame {count} - face detected in image {get_file_name(vid)}")
                        cv2.imwrite(out + "/frame%d.jpg" % count, face)
                    elif not frames_to_extract: # if there is no json file with the index of the frames to extract, extract face from all the frames with face detected
                        print(f"frame {count} - face detected in image {get_file_name(vid)}")
                        # save face
                        cv2.imwrite(out + "/frame%d.jpg" % count, face)
                    else:
                        print(f"frame {count} - face detected but frame not in the index of the frames to extract, skipping frame {count}")

            count += 1

        print("done")
        vidObj.release()
        cv2.destroyAllWindows()


def sanity_check_faces_extraction(faces_paths):
    """
    Sanity check to see if the number of frames extracted from the original videos is correct (390 frames)
    """
    print("\nsanity check: ")
    # add sanity check to see if the number of frames extracted from the videos is correct (390 frames)
    for sub in extract_subfolders(faces_paths):
        print("sub:", sub)
        # print(len(extract_files(sub)))
        if len(extract_files(sub)) != 390: # check if all frames extracte from the video
            print("ERROR: number of frames extracted from the real videos is not correct (!= 390)")
        else:
            print("number of frames extracted from the real videos is correct (= 390)")

In [ ]:
import json
import os

json_file_path = '../FOWS_demo/demo_videos/dlc/dlc_occ_frames.json'

# Initialize json_frames_occ to None, in case loading fails
json_frames_occ = None

# read the json file
if os.path.exists(json_file_path):
    try:
        with open(json_file_path, 'r') as f:
            # Read the entire content of the file first
            file_content = f.read()

            # Check if the file content is empty or only whitespace
            if not file_content.strip():
                print(f"Error: The file '{json_file_path}' is empty or contains only whitespace.")
            else:
                # Attempt to parse the content as JSON
                json_frames_occ = json.loads(file_content)
                print("Successfully loaded JSON content:")
                print(json_frames_occ)

    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e}")
        print(f"The file '{json_file_path}' contains malformed JSON. Please ensure it is correctly formatted.")
    except Exception as e:
        print(f"An unexpected error occurred while processing the file: {e}")
else:
    print(f"Error: The file '{json_file_path}' does not exist. Please check the path and ensure the file is present.")

# The variable 'json_frames_occ' will hold the loaded data or remain None if an error occurred.

Successfully loaded JSON content:
{'user_182545': {'hand_occlusion_1': ['frame170.jpg', 'frame171.jpg', 'frame172.jpg', 'frame173.jpg', 'frame174.jpg', 'frame175.jpg', 'frame176.jpg', 'frame177.jpg', 'frame178.jpg', 'frame179.jpg', 'frame180.jpg', 'frame181.jpg', 'frame182.jpg', 'frame183.jpg', 'frame184.jpg', 'frame185.jpg', 'frame186.jpg', 'frame187.jpg', 'frame188.jpg', 'frame189.jpg', 'frame190.jpg', 'frame191.jpg', 'frame192.jpg', 'frame193.jpg', 'frame194.jpg', 'frame195.jpg', 'frame196.jpg', 'frame197.jpg', 'frame198.jpg', 'frame199.jpg', 'frame200.jpg', 'frame201.jpg', 'frame202.jpg', 'frame203.jpg', 'frame204.jpg', 'frame205.jpg', 'frame206.jpg', 'frame207.jpg', 'frame208.jpg', 'frame209.jpg', 'frame210.jpg', 'frame211.jpg', 'frame212.jpg', 'frame213.jpg', 'frame214.jpg', 'frame215.jpg', 'frame216.jpg', 'frame217.jpg', 'frame218.jpg', 'frame219.jpg', 'frame220.jpg', 'frame221.jpg', 'frame222.jpg', 'frame223.jpg', 'frame224.jpg', 'frame225.jpg', 'frame226.jpg', 'frame227.jpg', 

In [ ]:
print("TEST DLC VIDEOS")
dlc_vid_root = '../FOWS_demo/demo_videos/dlc/'
dlc_occ_root = '../FOWS_demo/user_faces/dlc_occ/'
dlc_no_occ_root = '../FOWS_demo/user_faces/dlc_no_occ/'

json_occ = '../FOWS_demo/demo_videos/dlc/dlc_occ_frames.json'
json_no_occ = '../FOWS_demo/demo_videos/dlc/dlc_no_occ_frames.json'

# dlc_vid_root points to the directory containing user_XXXXXX folders
# dlc_faces_root points to the base directory where user_XXXXXX output folders will be created

TEST DLC VIDEOS


In [ ]:
# Assuming dlc_vid_root points to '../FOWS_demo/demo_videos/dlc/'
# and dlc_faces_root points to '../FOWS_demo/user_faces/dlc_occ/'

occ_video_paths = []
occ_output_paths = []
no_occ_video_paths = []
no_occ_output_paths = []

# Iterate through each user's video directory within dlc_vid_root
user_video_dirs = extract_subfolders(dlc_vid_root)

for user_video_dir in user_video_dirs:
    # Extract user_id from the user's video directory path
    user_id = os.path.basename(os.path.normpath(user_video_dir))

    # Construct the full output path for this specific user
    dlc_occ_faces_user_path = os.path.join(dlc_occ_root, user_id)
    dlc_no_occ_faces_user_path = os.path.join(dlc_no_occ_root, user_id)

    # Create the user-specific directory if it doesn't exist
    if not os.path.exists(dlc_occ_faces_user_path):
        print(f"Creating user directory: {dlc_occ_faces_user_path}")
        os.makedirs(dlc_occ_faces_user_path)
    else:
        print(f"User directory already exists: {dlc_occ_faces_user_path}")

    # Now create challenge subfolders inside the user directory
    print(f"Creating/Ensuring challenge subfolders in: {dlc_occ_faces_user_path}")
    create_subfolders(dlc_occ_faces_user_path)


    # Create the user-specific directory if it doesn't exist
    if not os.path.exists(dlc_no_occ_faces_user_path):
        print(f"Creating user directory: {dlc_no_occ_faces_user_path}")
        os.makedirs(dlc_no_occ_faces_user_path)
    else:
        print(f"User directory already exists: {dlc_no_occ_faces_user_path}")

    # Now create challenge subfolders inside the user directory
    print(f"Creating/Ensuring challenge subfolders in: {dlc_no_occ_faces_user_path}")
    create_subfolders(dlc_no_occ_faces_user_path)

    # Collect video paths for this user
    # Iterate through challenge subfolders within the user's video directory
    for challenge_video_dir in extract_subfolders(user_video_dir):
        videos_in_challenge = extract_files(challenge_video_dir)
        if videos_in_challenge:
            occ_video_paths.extend(videos_in_challenge)
            no_occ_video_paths.extend(videos_in_challenge)
            # print(videos_in_challenge)
            # Corresponding output paths for this user's challenge
            challenge_id = os.path.basename(os.path.normpath(challenge_video_dir))
            user_occ_output_path = os.path.join(dlc_occ_faces_user_path, challenge_id)
            user_no_occ_output_path = os.path.join(dlc_no_occ_faces_user_path, challenge_id)
            # Extend output paths with the correct user_challenge_output_path for each video in the challenge
            # Assuming one video per challenge, but if multiple, this needs adjustment
            occ_output_paths.extend([user_occ_output_path] * len(videos_in_challenge))
            no_occ_output_paths.extend([user_no_occ_output_path] * len(videos_in_challenge))

# Sort both lists to ensure they match correctly, assuming a consistent naming convention
occ_video_paths = sort_paths(occ_video_paths)
occ_output_paths = sort_paths(occ_output_paths)

print(occ_video_paths)
print(occ_output_paths)

no_occ_video_paths = sort_paths(no_occ_video_paths)
no_occ_output_paths = sort_paths(no_occ_output_paths)

print(no_occ_video_paths)
print(no_occ_output_paths)



# extract_faces_from_videos_json(all_video_paths, all_output_paths, json_file_path)

Creating user directory: /content/drive/MyDrive/Colab Notebooks/FOWS_demo/user_faces/dlc_occ/user_196133
Creating/Ensuring challenge subfolders in: /content/drive/MyDrive/Colab Notebooks/FOWS_demo/user_faces/dlc_occ/user_196133
Creating user directory: /content/drive/MyDrive/Colab Notebooks/FOWS_demo/user_faces/dlc_no_occ/user_196133
Creating/Ensuring challenge subfolders in: /content/drive/MyDrive/Colab Notebooks/FOWS_demo/user_faces/dlc_no_occ/user_196133
Creating user directory: /content/drive/MyDrive/Colab Notebooks/FOWS_demo/user_faces/dlc_occ/user_85808
Creating/Ensuring challenge subfolders in: /content/drive/MyDrive/Colab Notebooks/FOWS_demo/user_faces/dlc_occ/user_85808
Creating user directory: /content/drive/MyDrive/Colab Notebooks/FOWS_demo/user_faces/dlc_no_occ/user_85808
Creating/Ensuring challenge subfolders in: /content/drive/MyDrive/Colab Notebooks/FOWS_demo/user_faces/dlc_no_occ/user_85808
Creating user directory: /content/drive/MyDrive/Colab Notebooks/FOWS_demo/user_f

In [11]:
print("TEST DLC VIDEOS - OCC")
extract_faces_from_videos_json(occ_video_paths, occ_output_paths, json_occ)

TEST DLC VIDEOS - OCC
extracting faces from video: /content/drive/MyDrive/Colab Notebooks/FOWS_demo/demo_videos/dlc/user_85808/hand_occlusion_1/user_85808_hand_occlusion_1_DLC.mp4
saving faces to: /content/drive/MyDrive/Colab Notebooks/FOWS_demo/user_faces/dlc_occ/user_85808/hand_occlusion_1
user_id: user_85808 - algo: DLC - challenge_id: hand_occlusion_1
frames to extract from the json file for user user_85808, algorithm DLC, challenge hand_occlusion_1: [200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299]
[200, 201, 202, 203, 204, 205, 206, 207,

In [17]:
# extract_files([sub for sub in extract_subfolders('/content/drive/MyDrive/Colab Notebooks/FOWS_demo/user_faces/dlc_occ')])

TypeError: expected str, bytes or os.PathLike object, not list

In [ ]:
print("\nsanity check: ")
# add sanity check to see if the number of frames extracted from the original videos is correct (390 frames)
for user_subfolder in extract_subfolders('../FOWS_demo/user_faces/dlc_occ'):
    user_id_name = os.path.basename(user_subfolder)
    print(f"\nChecking user folder: {user_id_name}")
    for challenge_subfolder in extract_subfolders(user_subfolder):
        challenge_id_name = os.path.basename(challenge_subfolder)
        print(f"  Checking challenge: {challenge_id_name}")
        files_in_challenge = extract_files(challenge_subfolder)
        if files_in_challenge:
            num_files = len(files_in_challenge)
            print(f"    Number of frames extracted: {num_files}")
            if num_files != 100:
                print(f"    ERROR: number of frames extracted from the facedancer videos is not correct")
            else:
                print("    Number of frames extracted from the facedancer videos is correct")
        else:
            print("    No files found in this challenge folder.")


sanity check: 

Checking user folder: user_196133
  Checking challenge: hand_occlusion_1
    Number of frames extracted: 100
    Number of frames extracted from the facedancer videos is correct
  Checking challenge: hand_occlusion_2
    Number of frames extracted: 100
    Number of frames extracted from the facedancer videos is correct
  Checking challenge: hand_occlusion_3
    Number of frames extracted: 100
    Number of frames extracted from the facedancer videos is correct
  Checking challenge: obj_occlusion_1
    Number of frames extracted: 100
    Number of frames extracted from the facedancer videos is correct
  Checking challenge: obj_occlusion_2
    Number of frames extracted: 100
    Number of frames extracted from the facedancer videos is correct
  Checking challenge: obj_occlusion_3
    Number of frames extracted: 100
    Number of frames extracted from the facedancer videos is correct

Checking user folder: user_85808
  Checking challenge: hand_occlusion_1
    Number of f

In [20]:
print("TEST DLC VIDEOS - NO-OCC")
extract_faces_from_videos_json(no_occ_video_paths, no_occ_output_paths, json_no_occ)
print("done!")

TEST DLC VIDEOS - NO-OCC
extracting faces from video: /content/drive/MyDrive/Colab Notebooks/FOWS_demo/demo_videos/dlc/user_85808/hand_occlusion_1/user_85808_hand_occlusion_1_DLC.mp4
saving faces to: /content/drive/MyDrive/Colab Notebooks/FOWS_demo/user_faces/dlc_no_occ/user_85808/hand_occlusion_1
user_id: user_85808 - algo: DLC - challenge_id: hand_occlusion_1
frames to extract from the json file for user user_85808, algorithm DLC, challenge hand_occlusion_1: [0, 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 3, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 4, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 5, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 6, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 7, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 8, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 9, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
[0, 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 3, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 4, 40, 41, 

In [ ]:
print("\nsanity check: ")
# add sanity check to see if the number of frames extracted from the original videos is correct (390 frames)
for user_subfolder in extract_subfolders('../FOWS_demo/user_faces/dlc_no_occ'):
    user_id_name = os.path.basename(user_subfolder)
    print(f"\nChecking user folder: {user_id_name}")
    for challenge_subfolder in extract_subfolders(user_subfolder):
        challenge_id_name = os.path.basename(challenge_subfolder)
        print(f"  Checking challenge: {challenge_id_name}")
        files_in_challenge = extract_files(challenge_subfolder)
        if files_in_challenge:
            num_files = len(files_in_challenge)
            print(f"    Number of frames extracted: {num_files}")
            if num_files != 100:
                print(f"    ERROR: number of frames extracted from the facedancer videos is not correct")
            else:
                print("    Number of frames extracted from the facedancer videos is correct")
        else:
            print("    No files found in this challenge folder.")


sanity check: 

Checking user folder: user_196133
  Checking challenge: hand_occlusion_1
    Number of frames extracted: 100
    Number of frames extracted from the facedancer videos is correct
  Checking challenge: hand_occlusion_2
    Number of frames extracted: 100
    Number of frames extracted from the facedancer videos is correct
  Checking challenge: hand_occlusion_3
    Number of frames extracted: 100
    Number of frames extracted from the facedancer videos is correct
  Checking challenge: obj_occlusion_1
    Number of frames extracted: 100
    Number of frames extracted from the facedancer videos is correct
  Checking challenge: obj_occlusion_2
    Number of frames extracted: 100
    Number of frames extracted from the facedancer videos is correct
  Checking challenge: obj_occlusion_3
    Number of frames extracted: 100
    Number of frames extracted from the facedancer videos is correct

Checking user folder: user_85808
  Checking challenge: hand_occlusion_1
    Number of f